In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn

In [ ]:
!pip install fastai

In [ ]:
!pip install timm

In [ ]:
from fastai.vision.all import *

In [ ]:
from fastai.tabular.all import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
PALM_URL = "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/ccr8cm22vz-1.zip"
path = untar_data(PALM_URL)

In [ ]:
!pip install rarfile av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.8/33.8 MB 66.2 MB/s eta 0:00:00


In [ ]:
!pip install fastai2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.4/179.4 kB 3.4 MB/s eta 0:00:00


In [ ]:
from fastai.data import *

In [ ]:
!unrar x /root/.fastai/data/ccr8cm22vz-1/Palm.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /root/.fastai/data/ccr8cm22vz-1/Palm.rar

Creating    Palm                                                      OK
Extracting  Palm/Anemic-260 (10).png                                       0%  OK 
Extracting  Palm/Anemic-260 (11).png                                       0%  OK 
Extracting  Palm/Anemic-260 (12).png                                       0%  OK 
Extracting  Palm/Anemic-260 (2).png                                        0%  OK 
Extracting  Palm/Anemic-260 (3).png                                        0%  OK 
Extracting  Palm/Anemic-260 (4).png                                        0%  OK 
Extracting  Palm/Anemic-260 (5).png                                        0%  OK 
Extracting  Palm/Anemic-260 (6).png                                        0%  OK 
Extracting  Palm/Anemic-260 (7).png                      

In [ ]:
PALM_ROOT = "/content/Palm"
files = get_image_files(PALM_ROOT)
files[0], files[10]

(Path('/content/Palm/Non-AnemicP-118 (2).png'),
 Path('/content/Palm/AnemicP-224 (6).png'))

In [ ]:
path = Path(PALM_ROOT)
path

Path('/content/Palm')

In [ ]:
def is_anemic(f): return "non-anemic" not in f.lower()

In [ ]:
dls = ImageDataLoaders.from_name_func(path, files, is_anemic, item_tfms=Resize(128))

In [ ]:
cbc_data_path = '/content/drive/MyDrive/anemia.csv'
cbc_data = pd.read_csv(cbc_data_path)

In [ ]:
cat_names = ['Gender']
cont_names = ['Hemoglobin', 'MCH', 'MCHC', 'MCV']
y_names = 'Result'

procs = [Categorify, FillMissing, Normalize]

dls_tab = TabularDataLoaders.from_df(
    cbc_data,
    procs=procs,
    cat_names=cat_names,
    cont_names=cont_names,
    y_names=y_names,
    y_block=CategoryBlock(),
    splits=RandomSplitter()(range_of(cbc_data))
)

In [ ]:
class MultiModalModel(nn.Module):
    def __init__(self, image_model, tabular_input_size, num_classes):
        super().__init__()
        self.image_model = image_model
        self.tabular_model = nn.Sequential(
            nn.Linear(tabular_input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        self.fusion_layer = nn.Linear(image_model.num_features + 64, 256)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, image, tabular):
        img_features = self.image_model(image)
        tabular_features = self.tabular_model(tabular)
        combined = torch.cat([img_features, tabular_features], dim=1)
        fused = self.fusion_layer(combined)
        return self.classifier(fused)

In [ ]:
from timm import list_models
print(list_models('*convnext*'))

['convnext_atto', 'convnext_atto_ols', 'convnext_atto_rms', 'convnext_base', 'convnext_femto', 'convnext_femto_ols', 'convnext_large', 'convnext_large_mlp', 'convnext_nano', 'convnext_nano_ols', 'convnext_pico', 'convnext_pico_ols', 'convnext_small', 'convnext_tiny', 'convnext_tiny_hnf', 'convnext_xlarge', 'convnext_xxlarge', 'convnext_zepto_rms', 'convnext_zepto_rms_ols', 'convnextv2_atto', 'convnextv2_base', 'convnextv2_femto', 'convnextv2_huge', 'convnextv2_large', 'convnextv2_nano', 'convnextv2_pico', 'convnextv2_small', 'convnextv2_tiny', 'test_convnext', 'test_convnext2', 'test_convnext3']


In [ ]:
# Assuming 'dls' from your earlier code refers to the ImageDataLoaders object,
# you can simply assign it to dls_img:
dls_img = dls

In [ ]:
# Step 5: Train Image Model
learn_img = vision_learner(dls_img, 'convnext_tiny', metrics=error_rate, pretrained=True)
learn_img.fine_tune(10)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

epoch,train_loss,valid_loss,error_rate,time
0,0.878176,0.493482,0.220657,00:05


epoch,train_loss,valid_loss,error_rate,time
0,0.469788,0.245308,0.095070,00:06
1,0.344532,0.176419,0.070423,00:06
2,0.246325,0.149662,0.063380,00:06
3,0.180249,0.128919,0.045775,00:06
4,0.119425,0.062398,0.025822,00:06
5,0.082077,0.065644,0.019953,00:06
6,0.058145,0.056672,0.019953,00:06
7,0.032768,0.039326,0.015258,00:06
8,0.024378,0.045112,0.014085,00:06
9,0.018271,0.039714,0.015258,00:06


In [ ]:
# Step 6: Train Tabular Model
learn_tab = tabular_learner(dls_tab, metrics=accuracy)
learn_tab.fit_one_cycle(10, lr_max=1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,0.497236,0.498580,0.873239,00:00
1,0.358359,0.227529,0.929577,00:00
2,0.294754,0.192396,0.926056,00:00
3,0.250367,0.152642,0.954225,00:00
4,0.211719,0.100240,0.971831,00:00
5,0.178662,0.080818,0.975352,00:00
6,0.147496,0.058257,0.985915,00:00
7,0.122715,0.053353,0.985915,00:00
8,0.110929,0.049073,0.985915,00:00
9,0.097956,0.056958,0.982394,00:00


In [ ]:
# Step 6: Extract Independent Embeddings
# Extract image embeddings
image_dl = dls_img.test_dl(files)
image_preds, _ = learn_img.get_preds(dl=image_dl)
image_embeddings = image_preds.numpy()
# Extract CBC embeddings
cbc_preds, _ = learn_tab.get_preds(dl=dls_tab.train)
cbc_embeddings = cbc_preds.numpy()

In [ ]:
# Step 7: Handle Independent Datasets with Padding
image_dim = image_embeddings.shape[1]
cbc_dim = cbc_embeddings.shape[1]

# Match dimensions by padding smaller embedding
if len(image_embeddings) > len(cbc_embeddings):
    cbc_embeddings = np.pad(cbc_embeddings, ((0, len(image_embeddings) - len(cbc_embeddings)), (0, 0)))
elif len(image_embeddings) < len(cbc_embeddings):
    image_embeddings = np.pad(image_embeddings, ((0, len(cbc_embeddings) - len(image_embeddings)), (0, 0)))

# Combine features
combined_features = np.hstack([image_embeddings, cbc_embeddings])

# Labels (Ensure correct alignment; here using CBC labels as a placeholder)
labels = cbc_data['Result'].values
if len(labels) < len(combined_features):
    labels = np.pad(labels, (0, len(combined_features) - len(labels)))

In [ ]:
# Step 8: Train Fusion Classifier
X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
# Evaluate
y_pred = clf.predict(X_test)
print(f"Fusion Model Accuracy: {accuracy_score(y_test, y_pred)}")

Fusion Model Accuracy: 0.8098591549295775


In [ ]:
# Step 9: Conflict Handling Logic
# Example: Use probabilities from RandomForest to simulate confidence thresholds
image_prob = 0.8  # Placeholder for image-driven anemia probability
cbc_prob = 0.4    # Placeholder for CBC-driven anemia probability

if image_prob >= 0.7 and cbc_prob < 0.5:
    final_result = "Anemic (Image-driven)"
elif cbc_prob >= 0.7 and image_prob < 0.5:
    final_result = "Non-anemic (CBC-driven)"
else:
    final_result = "Uncertain (Conflicting inputs)"

print(f"Final Decision: {final_result}")

Final Decision: Anemic (Image-driven)


In [ ]:
# Step 10: Save Models
learn_img.export('/content/drive/MyDrive/image_model.pkl')
learn_tab.export('/content/drive/MyDrive/tabular_model.pkl')
import joblib
joblib.dump(clf, '/content/drive/MyDrive/fusion_model.pkl')

['/content/drive/MyDrive/fusion_model.pkl']

In [ ]:
# Save Fusion Model as PTH
fusion_model_path = '/content/drive/MyDrive/fusion_model.pth'
combined_features_tensor = torch.tensor(combined_features)
torch.save({'model_state_dict': clf, 'features': combined_features_tensor}, fusion_model_path)
print(f"Fusion model saved to {fusion_model_path}")

Fusion model saved to /content/drive/MyDrive/fusion_model.pth
